# ViHSD Mixture of Experts experiment

This notebook prepares a Colab runtime, but training and evaluation are run manually from shell commands. That keeps the workflow explicit: edit a command, run it, and reuse the resulting `run_id` when evaluating.

Use the command cells below to:
- mount Drive and install the project
- authenticate Hugging Face and W&B through Colab's built-in Secrets panel
- run a smoke test or a full experiment
- evaluate the saved checkpoint
- compare saved metrics in the final summary cell

Create the `HF_TOKEN` and `WANDB_API_KEY` secrets in Colab before running the authentication cell. The credentials are stored by the respective login tools and are not written to the notebook, an `.env` file, or the repository.

The repository code remains the source of truth. Each run saves its resolved configuration, checkpoint, metrics, and predictions.

## 1. Mount Google Drive

The YAML checkpoint path points to `/content/drive/MyDrive/ViHSD-MoE/checkpoints`. Drive must be mounted before training so `.safetensors` files persist after the Colab runtime ends.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Clone the GitHub repository and install dependencies

This notebook treats GitHub as the source of truth. Each runtime clones the latest `main` branch into `/content/moe-vihsd`, installs dependencies from that clone, and runs the scripts there.

In [ ]:
PROJECT_DIR = '/content/moe-vihsd'
REPOSITORY_URL = 'https://github.com/lngphgthao/moe-vihsd.git'

!rm -rf $PROJECT_DIR
!git clone --depth 1 --branch main $REPOSITORY_URL $PROJECT_DIR
%cd $PROJECT_DIR
%pip install -q -r requirements.txt

In [ ]:
import os
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')
if not hf_token:
    raise RuntimeError('Create a Colab Secret named HF_TOKEN before continuing.')
os.environ['HF_TOKEN'] = hf_token

wandb_api_key = userdata.get('WANDB_API_KEY')
if not wandb_api_key:
    raise RuntimeError('Create a Colab Secret named WANDB_API_KEY before continuing.')
os.environ['WANDB_API_KEY'] = wandb_api_key

import wandb
wandb.login(verify=True)
os.environ['CHECKPOINT_DIR'] = '/content/drive/MyDrive/ViHSD-MoE/checkpoints'
os.environ['RESULTS_DIR'] = '/content/drive/MyDrive/ViHSD-MoE/results'
print('Hugging Face and W&B authentication configured from Colab Secrets.')

## 3. Run training and evaluation manually

The next cells contain commands you run yourself. Change the command before executing it; there is no Python loop that starts training or evaluation for you.

### Quick smoke test

Use this first to verify the runtime, dataset access, and model configuration:

```bash
python train.py --config configs/vihsd.yaml --smoke-test --run-id smoke-check
python evaluate.py --config configs/vihsd.yaml --run-id smoke-check
```

### Full baseline run

Use a unique `--run-id` for every experiment:

```bash
python train.py --config configs/vihsd.yaml --no-smoke-test --run-id baseline-current-moe
python evaluate.py --config configs/vihsd.yaml --run-id baseline-current-moe
```

### Full run with overrides

Repeat `--set` for each YAML value you want to change. These changes apply only to this run and do not modify `configs/vihsd.yaml`:

```bash
python train.py \
  --config configs/vihsd.yaml \
  --no-smoke-test \
  --run-id stronger-moe-v1 \
  --set model.architecture=stronger_moe \
  --set model.num_experts=8 \
  --set model.top_k=2 \
  --set training.learning_rate=0.0001 \
  --set training.loss_type=focal
python evaluate.py --config configs/vihsd.yaml --run-id stronger-moe-v1
```

For a command cell that you can edit and execute directly, see the next two cells.

## 4. Training command

Edit the command below and run the cell manually. The command prints the run ID and writes the checkpoint and metrics for that run.

For repeated experiments, change `--run-id` and any `--set` values before running the cell. Do not reuse a run ID unless you intentionally want to replace or inspect that run.

In [ ]:
# Edit this command, then run the cell manually.
!python train.py --config configs/vihsd.yaml --no-smoke-test --run-id baseline-current-moe

# Example variant:
# !python train.py --config configs/vihsd.yaml --no-smoke-test --run-id stronger-moe-v1 --set model.architecture=stronger_moe --set model.num_experts=8 --set model.top_k=2 --set training.loss_type=focal

## 5. Evaluation command

Run evaluation manually after the training command finishes. Use the same `run_id` so evaluation loads that run's `resolved_config.yaml` and best checkpoint.

In [ ]:
# Use the run_id from the training command above.
!python evaluate.py --config configs/vihsd.yaml --run-id baseline-current-moe

# Or evaluate a checkpoint directly:
# !python evaluate.py --config configs/vihsd.yaml --checkpoint checkpoints/baseline-current-moe/vihsd_moe_best.safetensors

## 6. Command-line argument reference

The scripts expose these arguments. Run commands from the repository root (`/content/moe-vihsd` in Colab).

### `train.py`

```text
-h, --help
    Show the argument reference and exit.

--config PATH
    YAML configuration file. Default: configs/vihsd.yaml.

--set SECTION.KEY=VALUE
    Override one existing YAML value for this run. Repeat the option for multiple
    overrides. Values are parsed as YAML, so use true, false, null, numbers,
    quoted strings, or YAML lists when needed.

--smoke-test / --no-smoke-test
    Override the YAML smoke-test setting. --smoke-test uses training.smoke.epochs
    and training.smoke.max_train_samples; --no-smoke-test uses the normal
    training.epochs and training.max_train_samples values. Default: use the value
    from the YAML file.

--run-id ID
    Folder/name for this experiment. If omitted, a timestamp plus profile is
    generated automatically. Use a unique value for comparable experiments.
```

### `evaluate.py`

```text
-h, --help
    Show the argument reference and exit.

--config PATH
    YAML configuration file used when resolving the checkpoint. Default:
    configs/vihsd.yaml.

--checkpoint PATH
    Direct path to a .safetensors checkpoint. Takes precedence over --run-id.

--run-id ID
    Evaluate checkpoints/<ID>/vihsd_moe_best.safetensors. If omitted, evaluate
    the checkpoint recorded in checkpoints/latest_run.json.
```

### Useful `--set` keys

Any existing key in `configs/vihsd.yaml` can be overridden. Common experiment controls include:

| Section      | Parameter                          | Description / Allowed Values                            |
| :----------- | :--------------------------------- | :------------------------------------------------------ |
| **Model**    | `model.architecture`               | `current_moe` / `stronger_moe` / `pretrained_backbone`  |
|              | `model.num_experts`                | Number of experts                                       |
|              | `model.top_k`                      | Experts selected per token; `1 <= top_k <= num_experts` |
|              | `model.model_dim`                  | Model hidden size                                       |
|              | `model.expert_hidden_dim`          | Expert hidden size                                      |
|              | `model.num_layers`                 | Number of Transformer layers                            |
|              | `model.dropout`                    | Dropout rate                                            |
| **Training** | `training.epochs`                  | Number of full-profile epochs                           |
|              | `training.batch_size`              | Batch size                                              |
|              | `training.learning_rate`           | Optimizer learning rate                                 |
|              | `training.weight_decay`            | Optimizer weight decay                                  |
|              | `training.loss_type`               | `cross_entropy` / `focal`                               |
|              | `training.focal_gamma`             | Focal-loss gamma                                        |
|              | `training.class_weights`           | YAML list or `null`                                     |
|              | `training.max_train_samples`       | Sample limit or `null`                                  |
|              | `training.num_workers`             | DataLoader workers; `0` is safest in Colab              |
| **General**  | `seed`                             | Random seed                                             |
| **Routing**  | `routing.load_balance_loss_factor` | MoE load-balancing loss weight                          |



Paths and logging can also be overridden, for example:

```bash
python train.py --config configs/vihsd.yaml \
  --set paths.checkpoint_dir=/content/drive/MyDrive/ViHSD-MoE/checkpoints \
  --set paths.results_dir=/content/drive/MyDrive/ViHSD-MoE/results \
  --set logging.use_wandb=false
```

The scripts also honor the environment variables `CHECKPOINT_DIR` and `RESULTS_DIR`, which are configured in the setup cell above.

In [ ]:
import json
from pathlib import Path

# Set this to the run_id you just evaluated.
RUN_ID = 'baseline-current-moe'
metrics_path = Path('results') / RUN_ID / 'run_metrics.json'

if not metrics_path.exists():
    print(f'No saved metrics found at {metrics_path}')
else:
    metrics = json.loads(metrics_path.read_text(encoding='utf-8'))
    print(f'Evaluation summary: {RUN_ID}')
    display({
        'loss': metrics.get('test', {}).get('loss'),
        'accuracy': metrics.get('test', {}).get('accuracy'),
        'macro_f1': metrics.get('test', {}).get('macro_f1'),
        'weighted_f1': metrics.get('test', {}).get('weighted_f1'),
    })